# Notebook 7: Cross-Scenario Evaluation

Evaluate attacks and defenses across all available scenarios.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from helpers import (load_scenario, get_all_detections, get_ground_truth,
                     CameraAttackerDF, PointCloudAttackerDF, FusionAttackerDF,
                     DefensePipelineDF, CertifiedDefenseDF)
from attacks.camera_attacks import AttackType
from attacks.radar_lidar_attacks import PointCloudAttackType
from attacks.fusion_attacks import FusionAttackType
from defenses.defense_mechanisms import DefenseType

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)

## 7.1 Discover Available Scenarios

In [ ]:
import os
from pathlib import Path

data_dir = Path('../data/sensor_fusion_dataset')
scenarios = [d.name for d in data_dir.iterdir() if d.is_dir() and d.name.startswith('scenario')]
scenarios = sorted(scenarios)
print('Available scenarios:', scenarios)

## 7.2 Run Camera Attacks Across Scenarios

In [ ]:
cam = CameraAttackerDF(epsilon=0.05)
attack_types = [AttackType.FGSM, AttackType.PGD, AttackType.BIM]

results = []
for scenario in scenarios:
    try:
        loader = load_scenario(scenario)
        dets = get_all_detections(loader)
        gt = get_ground_truth(loader)
        
        if 3 not in dets:
            continue
        
        for atype in attack_types:
            attacked = cam.attack_detections(dets[3].copy(), atype, sensor_id=3)
            
            # Simple metric: mean bearing shift
            b_orig = dets[3]['bearing'].dropna().values
            b_att = attacked['bearing'].dropna().values
            if len(b_orig) > 0 and len(b_att) > 0:
                shift = np.mean(np.abs(b_att[:len(b_orig)] - b_orig[:len(b_att)]))
            else:
                shift = 0
            
            results.append({
                'scenario': scenario,
                'attack': atype.name,
                'bearing_shift': shift,
                'n_detections': len(attacked)
            })
    except Exception as e:
        print('Error in', scenario + ':', str(e))

results_df = pd.DataFrame(results)
print(results_df.head(10))

## 7.3 Visualize Cross-Scenario Results

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

for attack in results_df['attack'].unique():
    subset = results_df[results_df['attack'] == attack]
    ax.plot(subset['scenario'], subset['bearing_shift'], 'o-', label=attack, linewidth=2, markersize=8)

ax.set_xlabel('Scenario')
ax.set_ylabel('Mean Bearing Shift (rad)')
ax.set_title('Camera Attack Impact Across Scenarios')
ax.legend()
ax.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7.4 Defense Effectiveness Across Scenarios

In [ ]:
cert = CertifiedDefenseDF(certified_radius=0.05)
defense_results = []

for scenario in scenarios:
    try:
        loader = load_scenario(scenario)
        dets = get_all_detections(loader)
        gt = get_ground_truth(loader)
        
        if 3 not in dets:
            continue
        
        attacked = cam.attack_detections(dets[3].copy(), AttackType.FGSM, sensor_id=3)
        defended = cert.defend({'3': attacked.copy()}, gt)
        
        b_orig = dets[3]['bearing'].dropna().values
        b_att = attacked['bearing'].dropna().values
        b_def = defended['3']['bearing'].dropna().values
        
        if len(b_orig) > 0 and len(b_att) > 0:
            shift_att = np.mean(np.abs(b_att[:len(b_orig)] - b_orig[:len(b_att)]))
        else:
            shift_att = 0
        
        if len(b_orig) > 0 and len(b_def) > 0:
            shift_def = np.mean(np.abs(b_def[:len(b_orig)] - b_orig[:len(b_def)]))
        else:
            shift_def = 0
        
        defense_results.append({
            'scenario': scenario,
            'attack_shift': shift_att,
            'defense_shift': shift_def,
            'recovery': 1 - shift_def / max(shift_att, 1e-6)
        })
    except Exception as e:
        print('Error in', scenario + ':', str(e))

defense_df = pd.DataFrame(defense_results)
print(defense_df)

## 7.5 Recovery Rate Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(defense_df))
width = 0.35

ax.bar(x - width/2, defense_df['attack_shift'], width, label='Attack Shift', color='red')
ax.bar(x + width/2, defense_df['defense_shift'], width, label='Defense Shift', color='green')

ax.set_xlabel('Scenario')
ax.set_ylabel('Mean Bearing Shift (rad)')
ax.set_title('Defense Recovery: Attack vs Defense Shift')
ax.set_xticks(x)
ax.set_xticklabels(defense_df['scenario'], rotation=45)
ax.legend()
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()

# Recovery rate
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(defense_df['scenario'], defense_df['recovery'], color='steelblue')
ax.set_ylabel('Recovery Rate')
ax.set_title('Certified Defense Recovery Rate by Scenario')
ax.set_ylim(0, 1)
ax.grid(True, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()